# term

> Host terminal ptys on the gateway alongside kernels

In [ ]:
#| default_exp term

A terminal for a remote kernel needs to run on the same machine, in the same container and filesystem. Spawning the shell in a client process puts it in the wrong environment. Solveit's embedded terminal and ipyai's shell mode need a gateway connection for this reason.

Jupygate hosts named terminals alongside its kernels. Clients can list, create, and delete terminals through [Jupyter's terminals REST API](https://github.com/jupyter-server/jupyter_server_terminals). They attach to a running terminal over a websocket. Reconnecting after a page refresh or dropped connection resumes the same shell and replays recent scrollback.

[Ptymini](https://github.com/AnswerDotAI/ptymini) manages the pty sessions with `PtySession` and `PtyRegistry`. Those classes originally lived here. This module translates REST requests and websocket frames into ptymini calls. It checks authentication and applies user privileges through `_sudo`. The design notes and ipyai's stream-parsing requirements are in `meta/TERM.md`.

The websocket sends pty bytes unchanged in both directions. Text frames contain JSON control messages. This differs from [terminado](https://github.com/jupyter/terminado), which sends decoded terminal text inside JSON. Ipyai identifies command boundaries by parsing escape sequences in the byte stream. Decoding and re-encoding could alter those bytes. The gateway leaves them intact.


In [ ]:
#| export
import asyncio, json
from starlette.responses import JSONResponse, Response
from starlette.routing import Route, WebSocketRoute
from ptymini.core import PtyRegistry, Gap
from jupygate.core import _authed, _sudo


In [ ]:
import httpx, json, os, time
from jupygate.core import create_app, serve
from fastcore.test import test_eq, expect_fail
from websockets.sync.client import connect as ws_connect
from websockets.exceptions import InvalidStatus

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')


## The routes

`term_routes` returns Starlette routes backed by one `PtyRegistry`. The REST endpoints follow Jupyter's API. The websocket framing is specific to jupygate.

The routes use the kernels API's authentication check. Unauthorized requests return HTTP `403`. After authentication, a request for a missing terminal returns HTTP `404`. Websocket requests receive these errors during the opening handshake, without accepting a session.

Create a terminal with POST before attaching. Creation accepts options such as `argv`, `rc`, and `env`. A websocket connection never creates a shell. A misspelled terminal name must fail rather than start an unintended session.

When a creation request includes `username`, the route prefixes `_sudo(username)` to the command. User privileges are gateway policy. Ptymini receives the resulting command without needing to know about usernames.

Clients send keystrokes in binary frames. The server returns pty output in binary frames. Both preserve the original bytes. Text frames carry these JSON messages:

- The client sends `{"type": "set_size", "rows": R, "cols": C}` to resize the terminal.
- The server sends `{"type": "setup", "name": N}` after accepting the connection. Binary scrollback follows.
- The server sends `{"type": "gap", "bytes": N}` when the client has fallen behind the replay buffer and lost `N` bytes. The client needs to handle incomplete output, for example by clearing and redrawing.
- The server sends `{"type": "eof", "code": C}` when the pty exits.


In [ ]:
#| export
def term_routes(terminals:PtyRegistry, auth_token:str|None=None)->list:
    "Starlette routes for the terminals API, against one ptymini `PtyRegistry`."
    async def list_terms(request): return JSONResponse([t.model() for t in terminals.values()])

    async def create_term(request):
        body = await request.json() if await request.body() else {}
        kw = {k: body[k] for k in ('name','argv','cwd','env','appendenv','rc','rows','cols') if k in body}
        if body.get('username'): kw['argv'] = [*_sudo(body['username']), *(kw.get('argv') or terminals.argv)]
        try: t = await terminals.create(**kw)
        except Exception as e: return JSONResponse(dict(message=f'terminal failed to start: {e}'), status_code=500)
        return JSONResponse(t.model(), status_code=201)

    async def get_term(request): return JSONResponse(terminals.terms[request.path_params['name']].model())

    async def delete_term(request):
        await terminals.delete(request.path_params['name'])
        return Response(status_code=204)

    async def channel(ws):
        if not _authed(ws, auth_token): return await ws.send_denial_response(JSONResponse(dict(message='forbidden'), status_code=403))
        t = terminals.get(ws.path_params['name'])
        if t is None: return await ws.send_denial_response(JSONResponse(dict(message='no such terminal'), status_code=404))
        await ws.accept()
        await ws.send_text(json.dumps(dict(type='setup', name=t.name)))
        async def pump():
            async for item in t.attach(gaps=True):
                if isinstance(item, Gap): await ws.send_text(json.dumps(dict(type='gap', bytes=int(item))))
                else: await ws.send_bytes(item)
            await ws.send_text(json.dumps(dict(type='eof', code=t.exit_code)))
        task = asyncio.create_task(pump())
        try:
            while True:
                event = await ws.receive()
                if event['type'] == 'websocket.disconnect': break
                if event.get('bytes'): t.write(event['bytes'])
                elif event.get('text'):
                    c = json.loads(event['text'])
                    if c.get('type') == 'set_size': t.resize(c['rows'], c['cols'])
        finally: task.cancel()

    def guard(fn):
        async def inner(request):
            if not _authed(request, auth_token): return JSONResponse(dict(message='forbidden'), status_code=403)
            try: return await fn(request)
            except KeyError: return JSONResponse(dict(message='no such terminal'), status_code=404)
        return inner

    r = lambda p,meth,f: Route('/api/terminals'+p, guard(f), methods=[meth])
    return [r('','GET',list_terms), r('','POST',create_term), r('/{name}','GET',get_term),
        r('/{name}','DELETE',delete_term), WebSocketRoute('/api/terminals/{name}/channel', channel)]

## A live gateway

`create_app` mounts the terminal routes beside the kernel routes. Both use the same server, port, and authentication token.

Let's start a disposable local gateway. It has no terminals initially. The POST request creates a real Bash shell, which then appears in the terminal list:

In [ ]:
server = serve(create_app(), port=0, in_thread=True)
http = httpx.Client(base_url=server.url, timeout=30)
test_eq(http.get('/api/terminals').json(), [])
model = http.post('/api/terminals', json=dict(argv=BASH, env=BENV)).json()
test_eq(http.get('/api/terminals').json()[0]['name'], model['name'])
model

{'name': '1', 'alive': True, 'last_activity': 1788943430.633226}

The websocket starts with a `setup` text frame. We send a command as bytes and wait for its output. `ws_until` collects binary frames until they contain the requested pattern. It skips text control frames. This is the synchronous-websocket counterpart of `read_until`.

In [ ]:
def ws_until(ws, pat:bytes, timeout=10.0)->bytes:
    "Accumulate binary frames from sync websocket `ws` until `pat` appears (text frames are skipped)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        frame = ws.recv(timeout=end - time.monotonic())
        if isinstance(frame, bytes): buf += frame
    return buf

wsurl = f"{server.url.replace('http', 'ws')}/api/terminals/{model['name']}/channel"
ws = ws_connect(wsurl)
setup = json.loads(ws.recv(timeout=10))
test_eq(setup['type'], 'setup')
ws.send(b'echo wired $((2*3))\n')
out = ws_until(ws, b'wired 6')
out[-20:]

b' $((2*3))\r\nwired 6\r\n'

A second connection attaches to the same terminal. It receives `setup`, then the recent scrollback. That includes our earlier command's output even though this connection did not exist when the command ran. A browser refresh can recover its terminal display in the same way.

In [ ]:
ws2 = ws_connect(wsurl)
json.loads(ws2.recv(timeout=10))['type'], b'wired 6' in ws_until(ws2, b'wired 6')

('setup', True)

Send `set_size` as a text frame, then use `stty size` to check the shell's dimensions. Deleting the terminal returns HTTP `204` and stops its pty.

Each connection's output task sends `eof` when the pty ends. All attached clients receive it, not only the client that requested deletion. The example reads past any final binary output to check the control message:

After deletion, a new connection to the same URL gets HTTP `404`. Connecting cannot recreate the shell; that needs another POST.

In [ ]:
ws.send(json.dumps(dict(type='set_size', rows=50, cols=120)))
ws.send(b'stty size\n')
assert b'50 120' in ws_until(ws, b'50 120')
test_eq(http.delete(f"/api/terminals/{model['name']}").status_code, 204)
frame = ws.recv(timeout=10)
while isinstance(frame, bytes): frame = ws.recv(timeout=10)   # drain any final output; the text frame is the eof
test_eq(json.loads(frame)['type'], 'eof')
test_eq(http.get('/api/terminals').json(), [])
with expect_fail(InvalidStatus, 'HTTP 404'): ws_connect(wsurl)

In [ ]:
#| hide
ws.close()
ws2.close()
http.close()
server.should_exit = True


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()